# 01 — The naive map · *El mapa ingenuo*

### English
This is the module's opening demonstration: **how a heat map lies.** We take the
synthetic Davis fixtures — a known-answer street grid with a *planted* hotspot
(`seg-06`, low exposure, many reports) and a *busy decoy* (`seg-03`, "3rd St
(B–C)", the MOST raw reports but a boring rate) — and render two maps of the same
reports:

1. a **raw report-count** heat map (*the lie*), and
2. the **exposure-normalized rate** map (*the honest reading*).

Watch the busy decoy light up bright red on the left and dissolve on the right.
This is exactly the misuse the project's threat model calls **T4 — a naive
consumer misreading a raw-count map as danger** (`docs/THREAT-MODEL.md`).

> The Davis dataset is **synthetic demonstration data**, not real reports.

### Español
Esta es la demostración inicial del módulo: **cómo miente un mapa de calor.**
Tomamos los datos sintéticos de Davis — una cuadrícula de calles con respuesta
conocida, con un punto caliente *plantado* (`seg-06`, poca exposición, muchos
reportes) y un *señuelo concurrido* (`seg-03`, "3rd St (B–C)", la MAYOR cantidad
de reportes crudos pero una tasa modesta) — y dibujamos dos mapas de los mismos
reportes:

1. un mapa de calor de **conteo crudo de reportes** (*la mentira*), y
2. el mapa de **tasa normalizada por exposición** (*la lectura honesta*).

Observe cómo el señuelo concurrido se enciende en rojo intenso a la izquierda y
se disuelve a la derecha. Este es exactamente el mal uso que el modelo de
amenazas llama **T4 — un consumidor ingenuo que malinterpreta un mapa de conteo
crudo como peligro** (`docs/THREAT-MODEL.md`).

> El conjunto de datos de Davis son **datos sintéticos de demostración**, no
> reportes reales.

### Setup · *Preparación*
Load the demo config and run the full, deterministic pipeline — the same code
path `make demo` and the test suite use. No randomness: the fixtures are a fixed
known answer. · *Cargue la configuración y ejecute la tubería determinista — la
misma ruta de código que usan `make demo` y las pruebas. Sin aleatoriedad.*

In [ ]:
from pathlib import Path

from nearmiss.config import load_config
from nearmiss.engine import build_analysis


def repo_root() -> Path:
    """Walk up from the kernel's working directory to the repo root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate the nearmiss repo root")


ROOT = repo_root()
config = load_config(ROOT / "config" / "davis-demo.toml")
bundle = build_analysis(config)

names = {s.id: s.name for s in bundle.segments}
# The twelve ANALYZED blocks — the ones that carry exposure and reports. Every
# other block in the synthetic Davis grid is published as no-exposure context.
analyzed = [s for s in bundle.result.segments if s.exposure_estimate is not None]

print("city:", config.city, "(synthetic demonstration data — not real reports)")
print("exposure coverage:", round(bundle.result.exposure_coverage * 100), "%")
print("analyzed segments:", len(analyzed))

### The two readings of every segment · *Las dos lecturas de cada segmento*
For each analyzed block we have a **raw report count** (what a dot/heat map
counts) and an **exposure-normalized rate** with a 95% confidence interval (what
the honest analysis publishes). The decoy `seg-03` has the most raw reports; the
planted hotspot `seg-06` has the highest rate. · *Para cada bloque analizado
tenemos un **conteo crudo** (lo que cuenta un mapa de calor) y una **tasa
normalizada por exposición** con intervalo de confianza del 95% (lo que publica
el análisis honesto).*

In [ ]:
from IPython.display import Markdown

per = int(config.rate_per)
rows = [f"| Segment | Raw reports | Exposure | Rate /{per} | 95% CI |",
        "| --- | ---: | ---: | ---: | --- |"]
for s in sorted(analyzed, key=lambda s: -s.report_count):
    ci = f"{s.rate_ci_low:.2f}–{s.rate_ci_high:.2f}" if s.rate_ci_low is not None else "—"
    rows.append(
        f"| {names[s.segment_id]} | {s.report_count} | {s.exposure_estimate:.0f} "
        f"| {s.rate:.2f} | {ci} |"
    )
Markdown("\n".join(rows))

### A heat map with no plotting dependency · *Un mapa de calor sin dependencias de graficado*
`nearmiss` deliberately ships **no plotting library** — every figure is a
hand-built, byte-stable SVG (see `src/nearmiss/figures.py`). We do the same here
so this notebook runs anywhere Python runs and is reproducible byte-for-byte. The
helper below draws each analyzed street segment on a small geographic canvas,
colored by intensity on a light→dark "heat" ramp. Crucially — and per the threat
model's *"convey level in text and pattern, not color alone"* rule — every
segment also carries a **printed value** and a **line thickness** that scale with
intensity, so the encoding never relies on color. · *`nearmiss` no incluye
ninguna biblioteca de graficado; cada figura es un SVG estable byte a byte. El
ayudante siguiente dibuja cada segmento coloreado por intensidad, pero además el
valor va impreso y el grosor de línea escala con la intensidad: la codificación
nunca depende solo del color.*

In [ ]:
def _ramp(t: float) -> str:
    """Sequential light-yellow -> orange -> dark-red 'heat' ramp; t in [0, 1]."""
    stops = [(255, 255, 178), (253, 141, 60), (189, 0, 38)]
    t = 0.0 if t < 0.0 else 1.0 if t > 1.0 else t
    lo, hi, u = (stops[0], stops[1], t / 0.5) if t <= 0.5 else (stops[1], stops[2], (t - 0.5) / 0.5)
    r, g, b = (round(lo[i] + (hi[i] - lo[i]) * u) for i in range(3))
    return f"#{r:02x}{g:02x}{b:02x}"


def _coords():
    """{segment_id: ((lat, lon), (lat, lon))} for the analyzed blocks."""
    by_id = {s.id: s for s in bundle.segments}
    out = {}
    for s in analyzed:
        c = by_id[s.segment_id].coords
        out[s.segment_id] = (c[0], c[-1])
    return out


def _panel(seg_coords, values, title, x0, panel_w, panel_h, highlight):
    lats = [p[0] for pts in seg_coords.values() for p in pts]
    lons = [p[1] for pts in seg_coords.values() for p in pts]
    lat_min, lat_max = min(lats), max(lats)
    lon_min, lon_max = min(lons), max(lons)
    pad, top = 34, 40
    iw, ih = panel_w - 2 * pad, panel_h - top - pad

    def xy(lat, lon):
        fx = (lon - lon_min) / (lon_max - lon_min) if lon_max > lon_min else 0.5
        fy = (lat - lat_min) / (lat_max - lat_min) if lat_max > lat_min else 0.5
        return round(x0 + pad + fx * iw, 1), round(top + (1 - fy) * ih, 1)  # invert y

    vmax = max(values.values()) or 1.0
    parts = [
        f'<rect x="{x0}" y="0" width="{panel_w}" height="{panel_h}" fill="#fafafa" '
        f'stroke="#ccc"/>',
        f'<text x="{x0 + panel_w / 2:.0f}" y="24" text-anchor="middle" '
        f'font-family="sans-serif" font-size="14" font-weight="bold">{title}</text>',
    ]
    for sid in sorted(seg_coords):
        (la0, lo0), (la1, lo1) = seg_coords[sid]
        x1, y1 = xy(la0, lo0)
        x2, y2 = xy(la1, lo1)
        t = values[sid] / vmax
        width = round(3 + 9 * t, 1)  # thickness encodes intensity (not color alone)
        parts.append(
            f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{_ramp(t)}" '
            f'stroke-width="{width}" stroke-linecap="round"/>'
        )
        if sid in highlight:
            mx, my = round((x1 + x2) / 2, 1), round((y1 + y2) / 2, 1)
            parts.append(
                f'<circle cx="{mx}" cy="{my}" r="3" fill="none" stroke="#111" '
                f'stroke-width="1.2"/>'
                f'<text x="{mx}" y="{my - 8}" text-anchor="middle" font-family="sans-serif" '
                f'font-size="10" font-weight="bold">{highlight[sid]}: {values[sid]:.1f}</text>'
            )
    return "\n".join(parts)


def two_panel_heat(left_values, right_values, left_title, right_title, highlight):
    seg_coords = _coords()
    pw, ph, gap = 320, 300, 16
    w = pw * 2 + gap
    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{ph}" '
        f'viewBox="0 0 {w} {ph}" role="img" aria-label="{left_title} versus {right_title}, '
        f'{config.city} synthetic fixtures">',
        _panel(seg_coords, left_values, left_title, 0, pw, ph, highlight),
        _panel(seg_coords, right_values, right_title, pw + gap, pw, ph, highlight),
        "</svg>",
    ]
    return "\n".join(svg) + "\n"


print("heat-map helpers defined (hand-built SVG, no plotting dependency)")

### Left: the lie. Right: the honest reading. · *Izquierda: la mentira. Derecha: la lectura honesta.*
`seg-03` (the busy decoy) burns brightest on the **raw-count** map — it simply
has the most reports. Normalize by exposure and it **dissolves**; the planted
`seg-06` corridor emerges as the real hotspot. Same reports, opposite
conclusions. · *`seg-03` arde más intenso en el mapa de **conteo crudo**;
normalizado por exposición se **disuelve**, y emerge el corredor `seg-06` como el
verdadero punto caliente. Mismos reportes, conclusiones opuestas.*

In [ ]:
from IPython.display import SVG

raw = {s.segment_id: float(s.report_count) for s in analyzed}
rate = {s.segment_id: float(s.rate) for s in analyzed}
highlight = {"seg-03": "decoy 3rd St", "seg-06": "hotspot 5th St"}

SVG(two_panel_heat(raw, rate, "Raw report count (the lie)",
                   f"Exposure-normalized rate /{per} (honest)", highlight))

### The published ranking · *La clasificación publicada*
The honest artifact `nearmiss` actually publishes is not a colored surface at all
— it is a ranked table of exposure-normalized rates, each with a confidence
interval and an `n`, with the significant Getis-Ord Gi\* cluster marked in text
(`★`), never color alone. The busy decoy is nowhere near the top. · *El artefacto
honesto que `nearmiss` publica no es una superficie coloreada sino una tabla
ordenada de tasas normalizadas, cada una con intervalo de confianza y `n`, con el
grupo significativo Gi\* marcado en texto (`★`). El señuelo concurrido no está
cerca de la cima.*

In [ ]:
from nearmiss import figures

Markdown(figures.render_ranked_md(config))

### Takeaway · *Conclusión*
A raw-count heat map answers *"where do people report?"* and calls the answer
*danger*. The honest map answers *"where is the rate high, and is the cluster
statistically real?"* Label volume as volume; publish rates, intervals, and
significance; name the bias. That is the T4 mitigation — and the rest of this
module is about spotting the lie yourself and learning how fragile "significance"
can be. · *Un mapa de conteo crudo responde "¿dónde reporta la gente?" y llama a
eso peligro. El mapa honesto responde "¿dónde es alta la tasa y es real el
grupo?". Etiquete el volumen como volumen; publique tasas, intervalos y
significancia; nombre el sesgo. Esa es la mitigación de T4.*